# SFM Classifier Performance Benchmark

Analyzes `[PERF]` logs from AWS HealthOmics runs of the somatic featuremap classifier
to identify architectural bottlenecks and propose alternative frameworks for better performance.

**Pipeline stages under analysis:**
1. `filter_and_annotate_tr` -- VCF filtering + TR annotation (bcftools, bedtools)
2. `read_vcf_with_aggregation` -- VCF-to-Parquet (bcftools query | awk, ProcessPoolExecutor) + pileup features
3. `run_classifier` -- Polars-to-Pandas + XGBoost predict
4. `annotate_vcf_with_xgb_proba` -- DataFrame-to-TSV + bgzip + tabix + bcftools annotate

In [ ]:
import json
import os
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import boto3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

N_WORKERS = min(96, os.cpu_count() or 4)

## Section 1: Configuration and Log Fetching

Provide your HealthOmics run IDs below. Logs are cached under `/data/Runs/` so re-runs skip fetching.

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
RUN_IDS: list[str] = [
    "3448224",  # Tapas_SigGen_v1_27_0_pretrained_srsnv_models
    "7946896",
    "1550072",
    "3475679",
    "8639715",
    "1755479",
    "7975600",  # Tapas_SigGen_v1_27_0
    "7182497",
    "3616411",
    "7254722",
    "5372110",
    "3374534",
]

# Classifier task name pattern (to filter only classifier scatter tasks from run logs)
# Matches e.g. "SomaticFeaturemapClassifier-79-79080scattered"
CLASSIFIER_TASK_NAME_PATTERN = re.compile(r"SomaticFeaturemapClassifier")

AWS_REGION = "us-east-1"
LOG_CACHE_DIR = Path("/data/Runs/sfm_classifier_benchmark")
LOG_CACHE_DIR.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import threading

from botocore.config import Config as BotoConfig
from ugbio_omics.get_omics_log import OMICS_LOG_GROUP
from ugbio_omics.get_run_info import get_run_info

# CloudWatch GetLogEvents rate limit is ~10 TPS per account/region.
# Use a semaphore to cap concurrent API calls and retry with backoff on throttling.
CW_MAX_CONCURRENT = 5
CW_MAX_RETRIES = 8
_cw_semaphore = threading.Semaphore(CW_MAX_CONCURRENT)

_boto_config = BotoConfig(retries={"max_attempts": CW_MAX_RETRIES, "mode": "adaptive"})


def _fetch_single_task_log(run_id: str, task_id: str, task_name: str, output_dir: Path, region: str) -> Path | None:
    """Fetch a single task log from CloudWatch and save to disk. Returns path or None."""
    output_file = output_dir / f"run_{run_id}_task_{task_id}_{task_name}.log"
    if output_file.exists() and output_file.stat().st_size > 0:
        return output_file

    with _cw_semaphore:
        client = boto3.client("logs", region_name=region, config=_boto_config)
        log_stream = f"run/{run_id}/task/{task_id}"
        try:
            response = client.get_log_events(logGroupName=OMICS_LOG_GROUP, logStreamName=log_stream, startFromHead=True)
        except Exception as e:
            print(f"  [WARN] Failed to fetch {log_stream}: {e}")
            return None

        if not response.get("events"):
            return None

        with open(output_file, "w") as f:
            for event in response["events"]:
                f.write(f"{event['message']}\n")
            while response.get("events") and response.get("nextForwardToken"):
                response = client.get_log_events(
                    logGroupName=OMICS_LOG_GROUP,
                    logStreamName=log_stream,
                    nextToken=response["nextForwardToken"],
                    startFromHead=True,
                )
                for event in response["events"]:
                    f.write(f"{event['message']}\n")

    return output_file


def fetch_logs_for_run(run_id: str, region: str) -> list[dict]:
    """Fetch all task logs for a run (parallel per-task, rate-limited). Returns list of task info dicts."""
    output_dir = LOG_CACHE_DIR / run_id
    output_dir.mkdir(parents=True, exist_ok=True)

    omics_client = boto3.client("omics", region_name=region, config=_boto_config)
    run_info = get_run_info(run_id, client=omics_client)
    tasks = run_info.get("tasks", [])

    if not tasks:
        print(f"  Run {run_id}: no tasks found")
        return []

    results = []

    # Use many threads but the semaphore limits actual concurrent CloudWatch calls
    with ThreadPoolExecutor(max_workers=min(N_WORKERS, len(tasks))) as pool:
        futures = {}
        for task in tasks:
            tid, tname = task["taskId"], task["name"]
            fut = pool.submit(_fetch_single_task_log, run_id, tid, tname, output_dir, region)
            futures[fut] = task

        for fut in as_completed(futures):
            task = futures[fut]
            log_path = fut.result()
            results.append(
                {
                    "run_id": run_id,
                    "task_id": task["taskId"],
                    "task_name": task["name"],
                    "task_status": task.get("status", "UNKNOWN"),
                    "task_duration": task.get("duration"),
                    "log_path": log_path,
                }
            )

    return results


# Fetch logs for all runs (rate-limited via semaphore)
if not RUN_IDS:
    raise ValueError("RUN_IDS is empty -- paste your run IDs into the RUN_IDS list in the cell above.")

all_task_info: list[dict] = []

print(f"Fetching logs for {len(RUN_IDS)} runs (max {CW_MAX_CONCURRENT} concurrent CloudWatch calls)...")
with ThreadPoolExecutor(max_workers=min(N_WORKERS, len(RUN_IDS))) as pool:
    futures = {pool.submit(fetch_logs_for_run, rid, AWS_REGION): rid for rid in RUN_IDS}
    for fut in as_completed(futures):
        rid = futures[fut]
        try:
            task_infos = fut.result()
            all_task_info.extend(task_infos)
            n_logs = sum(1 for t in task_infos if t["log_path"] is not None)
            print(f"  Run {rid}: {n_logs}/{len(task_infos)} task logs fetched")
        except Exception as e:
            print(f"  Run {rid}: FAILED - {e}")

task_info_df = pd.DataFrame(all_task_info)
n_with_logs = task_info_df["log_path"].notna().sum() if "log_path" in task_info_df.columns else 0
print(f"\nTotal tasks: {len(task_info_df)}, with logs: {n_with_logs}")

# Fallback: if API failed, reconstruct task_info from cached log files on disk
if n_with_logs == 0:
    print("\nAPI fetch returned no logs — falling back to cached log files on disk...")
    cached_records = []
    log_filename_re = re.compile(r"run_(\d+)_task_(\d+)_(.+)\.log")
    for run_id in RUN_IDS:
        run_dir = LOG_CACHE_DIR / run_id
        if not run_dir.exists():
            continue
        for log_file in sorted(run_dir.glob("*.log")):
            m = log_filename_re.match(log_file.name)
            if m:
                cached_records.append(
                    {
                        "run_id": m.group(1),
                        "task_id": m.group(2),
                        "task_name": m.group(3),
                        "task_status": "COMPLETED",
                        "task_duration": None,
                        "log_path": log_file,
                    }
                )
    if cached_records:
        task_info_df = pd.DataFrame(cached_records)
        n_with_logs = task_info_df["log_path"].notna().sum()
        print(f"  Recovered {n_with_logs} cached log files across {task_info_df['run_id'].nunique()} runs")
    else:
        print("  No cached log files found either!")

task_info_df.head()

## Section 2: [PERF] Log Parser

Parses all `[PERF]` lines from fetched log files into a structured DataFrame.
Handles both the `_log_perf()` format (with `resources` nested JSON) and the inline
`_run_shell_command` format (direct JSON string).

In [ ]:
# Regex for [PERF] log lines:
#   [PERF] [stage:phase] message | {JSON}
# The JSON may or may not be valid (shell_command uses inline f-string formatting with None values)
PERF_REGEX = re.compile(r"\[PERF\]\s+\[([^\]]+)\]\s+(.+?)\s+\|\s+(\{.+\})\s*$")


def _safe_parse_json(raw: str) -> dict:
    """Parse JSON that may contain Python-style None/True/False literals."""
    cleaned = raw.replace(": None", ": null").replace(": True", ": true").replace(": False", ": false")
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"_raw_json": raw}


def _parse_stage_phase(location: str) -> tuple[str, str]:
    """Split 'stage:phase' into (stage, phase). If no colon, phase='info'."""
    if ":" in location:
        parts = location.split(":", 1)
        return parts[0], parts[1]
    return location, "info"


def parse_perf_lines_from_file(log_path: Path, run_id: str, task_name: str, task_id: str) -> list[dict]:
    """Extract all [PERF] events from a single log file."""
    records = []
    with open(log_path) as f:
        for line in f:
            m = PERF_REGEX.search(line)
            if not m:
                continue
            location, message, json_str = m.group(1), m.group(2), m.group(3)
            stage, phase = _parse_stage_phase(location)
            payload = _safe_parse_json(json_str)

            # Flatten nested 'resources' dict
            resources = payload.pop("resources", {})
            flat = {
                "run_id": run_id,
                "task_name": task_name,
                "task_id": task_id,
                "stage": stage,
                "phase": phase,
                "message": message,
                **payload,
                **{f"res_{k}": v for k, v in resources.items()},
            }
            records.append(flat)
    return records


def parse_all_perf_logs(task_info_df: pd.DataFrame) -> pd.DataFrame:
    """Parse [PERF] logs from all task log files using parallel workers."""
    rows_with_logs = task_info_df[task_info_df["log_path"].notna()].copy()
    all_records: list[dict] = []

    def _parse_one(row):
        return parse_perf_lines_from_file(Path(row["log_path"]), row["run_id"], row["task_name"], row["task_id"])

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(_parse_one, row): idx for idx, row in rows_with_logs.iterrows()}
        for fut in as_completed(futures):
            all_records.extend(fut.result())

    result_df = pd.DataFrame(all_records)
    if result_df.empty:
        print("WARNING: No [PERF] events found in any log file!")
        return result_df

    # Convert numeric columns
    numeric_cols = [
        "elapsed_sec",
        "mem_mb",
        "mem_delta_mb",
        "mem_peak_delta_mb",
        "input_size_mb",
        "output_size_mb",
        "vcf_size_mb",
        "parquet_size_mb",
        "output_vcf_size_mb",
        "input_vcf_size_mb",
        "variant_count",
        "row_count",
        "col_count",
        "n_threads",
        "postproc_elapsed_sec",
        "predict_elapsed_sec",
        "throughput_regions_per_sec",
        "total_part_size_mb",
        "res_rss_mb",
        "res_cpu_user_sec",
        "res_cpu_system_sec",
        "res_io_read_mb",
        "res_io_write_mb",
        "cpu_user_sec",
        "cpu_system_sec",
        "io_read_mb",
        "io_write_mb",
    ]
    for col in numeric_cols:
        if col in result_df.columns:
            result_df[col] = pd.to_numeric(result_df[col], errors="coerce")

    return result_df


perf_df = parse_all_perf_logs(task_info_df)
n_events = len(perf_df)
n_tasks = perf_df["task_id"].nunique()
n_runs = perf_df["run_id"].nunique()
print(f"Parsed {n_events} [PERF] events from {n_tasks} tasks across {n_runs} runs")
print(f"\nStages found: {sorted(perf_df['stage'].unique())}")
print(f"Phases found: {sorted(perf_df['phase'].unique())}")
perf_df.head(10)

In [ ]:
# Filter to classifier tasks only (ignore other WDL tasks like alignment, etc.)
classifier_tasks = task_info_df[task_info_df["task_name"].str.contains(CLASSIFIER_TASK_NAME_PATTERN, na=False)]
classifier_task_ids = set(classifier_tasks["task_id"])
print(f"Classifier tasks: {len(classifier_task_ids)} out of {len(task_info_df)} total tasks")
print(f"Classifier task names: {sorted(classifier_tasks['task_name'].unique())}")

# Filter perf_df to classifier tasks only
perf_df = perf_df[perf_df["task_id"].isin(classifier_task_ids)].copy()
print(f"\n[PERF] events after filtering: {len(perf_df)}")

# Show unique stages+phases for classifier tasks
stage_phase_counts = perf_df.groupby(["stage", "phase"]).size().reset_index(name="count")
stage_phase_counts.sort_values(["stage", "phase"])

## Section 3: Stage-Level Timing Breakdown

Which of the 4 pipeline stages dominates wall-clock time?

In [ ]:
# ── Extract exit-phase events for the 4 main stages ──────────────────────────
MAIN_STAGES = [
    "filter_and_annotate_tr",
    "read_vcf_with_aggregation",
    "run_classifier",
    "annotate_vcf_with_xgb_proba",
]

stage_exit_df = perf_df[(perf_df["stage"].isin(MAIN_STAGES)) & (perf_df["phase"] == "exit")].copy()

# Pivot: one row per (run_id, task_id), columns = stage elapsed_sec
timing_pivot = stage_exit_df.pivot_table(
    index=["run_id", "task_id", "task_name"],
    columns="stage",
    values="elapsed_sec",
    aggfunc="first",
).reset_index()

# Reorder columns
for s in MAIN_STAGES:
    if s not in timing_pivot.columns:
        timing_pivot[s] = 0.0

timing_pivot["total_sec"] = timing_pivot[MAIN_STAGES].sum(axis=1)

print(f"Tasks with stage timing: {len(timing_pivot)}")
timing_pivot.describe()[MAIN_STAGES + ["total_sec"]]

In [ ]:
# ── Pie chart: average time share per stage ───────────────────────────────────
avg_times = timing_pivot[MAIN_STAGES].mean()
fig_pie = px.pie(
    names=avg_times.index,
    values=avg_times.values,
    title="Average Time Share per Pipeline Stage",
    hole=0.35,
)
fig_pie.update_traces(textinfo="label+percent+value", texttemplate="%{label}<br>%{percent:.1%}<br>%{value:.1f}s")
fig_pie.show()

In [ ]:
# ── Box plot: per-stage elapsed time distribution across all shards ───────────
melted = timing_pivot.melt(
    id_vars=["run_id", "task_id", "task_name"],
    value_vars=MAIN_STAGES,
    var_name="stage",
    value_name="elapsed_sec",
)
fig_box = px.box(
    melted,
    x="stage",
    y="elapsed_sec",
    color="stage",
    title="Elapsed Time Distribution per Stage (across all scatter shards)",
    labels={"elapsed_sec": "Elapsed (sec)", "stage": "Pipeline Stage"},
    points="all",
)
fig_box.update_layout(showlegend=False, xaxis_tickangle=-25)
fig_box.show()

In [ ]:
# ── Stacked bar: per-shard timing breakdown (sample a subset for readability) ─
sample_size = min(60, len(timing_pivot))
timing_sample = timing_pivot.sample(n=sample_size, random_state=42).sort_values("total_sec", ascending=False)
timing_sample["label"] = timing_sample["run_id"].str[:6] + "/" + timing_sample["task_name"].str[-10:]

fig_stacked = go.Figure()
for stage in MAIN_STAGES:
    fig_stacked.add_trace(
        go.Bar(
            name=stage,
            x=timing_sample["label"],
            y=timing_sample[stage],
        )
    )
fig_stacked.update_layout(
    barmode="stack",
    title=f"Per-Shard Stage Timing Breakdown (top {sample_size} shards)",
    xaxis_title="Shard (run/task)",
    yaxis_title="Elapsed (sec)",
    xaxis_tickangle=-45,
    height=500,
)
fig_stacked.show()

In [ ]:
# ── Summary statistics table ──────────────────────────────────────────────────
stats = timing_pivot[MAIN_STAGES + ["total_sec"]].agg(["mean", "median", "std", "min", "max", "sum"])
stats.loc["pct_of_total"] = stats.loc["mean"] / stats.loc["mean"]["total_sec"] * 100
stats = stats.round(2)
print("Stage Timing Statistics (seconds):")
stats

## Section 4: Sub-Stage Deep Dive

Drill into each stage's internal operations to find the dominant sub-operation.

In [ ]:
# ── 4a: read_vcf_with_aggregation sub-stage breakdown ─────────────────────────
# Sub-stages: vcf_to_parquet, calculate_pileup_features, post-processing
# The exit event has: elapsed_sec (total), postproc_elapsed_sec
# vcf_to_parquet has its own exit with elapsed_sec
# calculate_pileup_features has its own end with elapsed_sec

sub_stages_read_vcf = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    row = {"run_id": run_id, "task_id": task_id}

    # Total stage time
    exit_evt = group[(group["stage"] == "read_vcf_with_aggregation") & (group["phase"] == "exit")]
    if exit_evt.empty:
        continue
    row["total_sec"] = exit_evt["elapsed_sec"].iloc[0]
    row["postproc_sec"] = exit_evt["postproc_elapsed_sec"].iloc[0] if "postproc_elapsed_sec" in exit_evt.columns else 0

    # vcf_to_parquet sub-stage (logged in classifier as vcf_to_parquet:end)
    vtp_end = group[(group["stage"] == "vcf_to_parquet") & (group["phase"] == "end")]
    if not vtp_end.empty:
        row["vcf_to_parquet_sec"] = vtp_end["elapsed_sec"].iloc[0]
    else:
        row["vcf_to_parquet_sec"] = 0

    # pileup calculation
    pileup_end = group[(group["stage"] == "calculate_pileup_features") & (group["phase"] == "end")]
    if not pileup_end.empty:
        row["pileup_sec"] = pileup_end["elapsed_sec"].iloc[0]
    else:
        row["pileup_sec"] = 0

    sub_stages_read_vcf.append(row)

sub_read_vcf_df = pd.DataFrame(sub_stages_read_vcf)
if not sub_read_vcf_df.empty:
    sub_read_vcf_df["other_sec"] = (
        sub_read_vcf_df["total_sec"]
        - sub_read_vcf_df["vcf_to_parquet_sec"]
        - sub_read_vcf_df["pileup_sec"]
        - sub_read_vcf_df["postproc_sec"]
    ).clip(lower=0)

    sub_cols = ["vcf_to_parquet_sec", "pileup_sec", "postproc_sec", "other_sec"]
    avg_sub = sub_read_vcf_df[sub_cols].mean()

    fig_sub_read = px.pie(
        names=avg_sub.index,
        values=avg_sub.values,
        title="read_vcf_with_aggregation: Average Sub-Stage Breakdown",
        hole=0.35,
    )
    fig_sub_read.update_traces(
        textinfo="label+percent+value", texttemplate="%{label}<br>%{percent:.1%}<br>%{value:.1f}s"
    )
    fig_sub_read.show()

    print("\nread_vcf_with_aggregation sub-stage stats (seconds):")
    sub_read_vcf_df[["total_sec"] + sub_cols].describe().round(2)

In [ ]:
# ── 4b: vcf_to_parquet internal breakdown ─────────────────────────────────────
# Sub-stages: _run_region_jobs (parallel bcftools+awk), parquet merge
# From featuremap_to_dataframe.py: _run_region_jobs:exit, vcf_to_parquet:merge_end

sub_stages_vtp = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    row = {"run_id": run_id, "task_id": task_id}

    # vcf_to_parquet total
    vtp_exit = group[(group["stage"] == "vcf_to_parquet") & (group["phase"] == "exit")]
    if vtp_exit.empty:
        continue
    row["total_sec"] = vtp_exit["elapsed_sec"].iloc[0]

    # Region jobs (parallel bcftools+awk processing)
    rj_exit = group[(group["stage"] == "_run_region_jobs") & (group["phase"] == "exit")]
    if not rj_exit.empty:
        row["region_jobs_sec"] = rj_exit["elapsed_sec"].iloc[0]
        row["region_count"] = rj_exit["region_count"].iloc[0] if "region_count" in rj_exit.columns else None
        row["throughput_regions_per_sec"] = (
            rj_exit["throughput_regions_per_sec"].iloc[0] if "throughput_regions_per_sec" in rj_exit.columns else None
        )
    else:
        row["region_jobs_sec"] = 0

    # Parquet merge
    merge_end = group[(group["stage"] == "vcf_to_parquet") & (group["phase"] == "merge_end")]
    if not merge_end.empty:
        row["merge_sec"] = merge_end["elapsed_sec"].iloc[0]
    else:
        row["merge_sec"] = 0

    # Config info (jobs count)
    config_evt = group[(group["stage"] == "vcf_to_parquet") & (group["phase"] == "config")]
    if not config_evt.empty and "jobs" in config_evt.columns:
        row["jobs"] = config_evt["jobs"].iloc[0]

    row["other_sec"] = max(0, row["total_sec"] - row["region_jobs_sec"] - row["merge_sec"])
    sub_stages_vtp.append(row)

sub_vtp_df = pd.DataFrame(sub_stages_vtp)
if not sub_vtp_df.empty:
    sub_cols = ["region_jobs_sec", "merge_sec", "other_sec"]
    avg_sub = sub_vtp_df[sub_cols].mean()

    fig_sub_vtp = px.pie(
        names=["region_jobs (bcftools+awk)", "parquet_merge", "other (setup/teardown)"],
        values=avg_sub.values,
        title="vcf_to_parquet: Average Sub-Stage Breakdown",
        hole=0.35,
    )
    fig_sub_vtp.update_traces(
        textinfo="label+percent+value", texttemplate="%{label}<br>%{percent:.1%}<br>%{value:.1f}s"
    )
    fig_sub_vtp.show()

    print("\nvcf_to_parquet sub-stage stats (seconds):")
    display_cols = ["total_sec"] + sub_cols
    if "region_count" in sub_vtp_df.columns:
        display_cols.append("region_count")
    if "throughput_regions_per_sec" in sub_vtp_df.columns:
        display_cols.append("throughput_regions_per_sec")
    sub_vtp_df[display_cols].describe().round(2)

In [ ]:
# ── 4c: run_classifier sub-stage breakdown ────────────────────────────────────
# Sub-stages: pandas_convert, model_load, predict
sub_stages_clf = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    row = {"run_id": run_id, "task_id": task_id}

    clf_exit = group[(group["stage"] == "run_classifier") & (group["phase"] == "exit")]
    if clf_exit.empty:
        continue
    row["total_sec"] = clf_exit["elapsed_sec"].iloc[0]
    row["predict_sec"] = clf_exit["predict_elapsed_sec"].iloc[0] if "predict_elapsed_sec" in clf_exit.columns else 0
    row["variant_count"] = clf_exit["variant_count"].iloc[0] if "variant_count" in clf_exit.columns else None
    row["mem_delta_mb"] = clf_exit["mem_delta_mb"].iloc[0] if "mem_delta_mb" in clf_exit.columns else None

    pandas_evt = group[(group["stage"] == "run_classifier") & (group["phase"] == "pandas_convert")]
    if not pandas_evt.empty:
        row["pandas_convert_sec"] = pandas_evt["elapsed_sec"].iloc[0]
        row["pandas_mem_mb"] = pandas_evt["mem_mb"].iloc[0] if "mem_mb" in pandas_evt.columns else None
    else:
        row["pandas_convert_sec"] = 0

    model_evt = group[(group["stage"] == "run_classifier") & (group["phase"] == "model_load")]
    if not model_evt.empty:
        row["model_load_sec"] = model_evt["elapsed_sec"].iloc[0]
    else:
        row["model_load_sec"] = 0

    row["other_sec"] = max(0, row["total_sec"] - row["pandas_convert_sec"] - row["model_load_sec"] - row["predict_sec"])
    sub_stages_clf.append(row)

sub_clf_df = pd.DataFrame(sub_stages_clf)
if not sub_clf_df.empty:
    sub_cols = ["pandas_convert_sec", "model_load_sec", "predict_sec", "other_sec"]
    avg_sub = sub_clf_df[sub_cols].mean()

    fig_sub_clf = px.pie(
        names=["Polars→Pandas convert", "model_load", "XGBoost predict", "other"],
        values=avg_sub.values,
        title="run_classifier: Average Sub-Stage Breakdown",
        hole=0.35,
    )
    fig_sub_clf.update_traces(
        textinfo="label+percent+value", texttemplate="%{label}<br>%{percent:.1%}<br>%{value:.1f}s"
    )
    fig_sub_clf.show()

    print("\nrun_classifier sub-stage stats (seconds):")
    sub_clf_df[["total_sec"] + sub_cols + ["variant_count", "mem_delta_mb"]].describe().round(2)

In [ ]:
# ── 4d: filter_and_annotate_tr sub-stage breakdown ────────────────────────────
# Sub-stages: view_vcf, bedtools closest pipeline + tabix, annotate_vcf

sub_stages_filter = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    row = {"run_id": run_id, "task_id": task_id}

    # Total filter_and_annotate_tr time
    filter_exit = group[(group["stage"] == "filter_and_annotate_tr") & (group["phase"] == "exit")]
    if filter_exit.empty:
        continue
    row["total_sec"] = filter_exit["elapsed_sec"].iloc[0]

    # view_vcf sub-stage (bcftools view + index)
    view_exit = group[(group["stage"] == "view_vcf") & (group["phase"] == "exit")]
    if not view_exit.empty:
        row["view_vcf_sec"] = view_exit["elapsed_sec"].iloc[0]
    else:
        row["view_vcf_sec"] = 0

    # annotate_vcf sub-stage (bcftools annotate + index) -- only the one within filter_and_annotate_tr
    # There may be multiple annotate_vcf events if annotate_vcf_with_xgb_proba also uses it;
    # take the first one (which corresponds to TR annotation)
    annot_vcf_exit = group[(group["stage"] == "annotate_vcf") & (group["phase"] == "exit")]
    if not annot_vcf_exit.empty:
        row["annotate_vcf_sec"] = annot_vcf_exit["elapsed_sec"].iloc[0]
    else:
        row["annotate_vcf_sec"] = 0

    # The remaining time is bedtools closest pipeline + tabix + overhead
    row["bedtools_pipeline_sec"] = max(0, row["total_sec"] - row["view_vcf_sec"] - row["annotate_vcf_sec"])
    sub_stages_filter.append(row)

sub_filter_df = pd.DataFrame(sub_stages_filter)
if not sub_filter_df.empty:
    sub_cols = ["view_vcf_sec", "bedtools_pipeline_sec", "annotate_vcf_sec"]
    avg_sub = sub_filter_df[sub_cols].mean()

    fig_sub_filter = px.pie(
        names=["bcftools view + index", "bedtools closest pipeline + tabix", "bcftools annotate + index"],
        values=avg_sub.values,
        title="filter_and_annotate_tr: Average Sub-Stage Breakdown",
        hole=0.35,
    )
    fig_sub_filter.update_traces(
        textinfo="label+percent+value", texttemplate="%{label}<br>%{percent:.1%}<br>%{value:.1f}s"
    )
    fig_sub_filter.show()

    print("\nfilter_and_annotate_tr sub-stage stats (seconds):")
    sub_filter_df[["total_sec"] + sub_cols].describe().round(2)

In [ ]:
# ── 4e: Shell command breakdown (filter_and_annotate_tr + annotate_vcf) ───────
# _run_shell_command events: :start has "cmd", :end has timing metrics.
# Propagate cmd from :start to :end by pairing them sequentially within each task.
shell_cmds = perf_df[perf_df["stage"] == "_run_shell_command"].copy()

if not shell_cmds.empty:
    # Propagate cmd from :start events to their corresponding :end events
    shell_cmds = shell_cmds.sort_index()
    for (_run_id, _task_id), group in shell_cmds.groupby(["run_id", "task_id"]):
        starts = group[group["phase"] == "start"]
        ends = group[group["phase"] == "end"]
        for (_s_idx, s_row), (e_idx, _) in zip(starts.iterrows(), ends.iterrows(), strict=False):
            if "cmd" in shell_cmds.columns and pd.notna(s_row.get("cmd")):
                shell_cmds.loc[e_idx, "cmd"] = s_row["cmd"]

    def _extract_tool(cmd_str):
        if pd.isna(cmd_str):
            return "unknown"
        cmd = str(cmd_str)
        if "bedtools" in cmd:
            return "bedtools closest pipeline"
        for tool in ["bcftools", "bgzip", "tabix", "sort", "cut"]:
            if tool in cmd:
                return tool
        return cmd.split()[0] if cmd.strip() else "unknown"

    shell_cmds["tool"] = shell_cmds["cmd"].apply(_extract_tool)
    shell_end = shell_cmds[shell_cmds["phase"] == "end"].copy()

    if not shell_end.empty:
        tool_stats = (
            shell_end.groupby("tool")
            .agg(
                count=("elapsed_sec", "count"),
                mean_elapsed=("elapsed_sec", "mean"),
                total_elapsed=("elapsed_sec", "sum"),
                mean_cpu_user=("cpu_user_sec", "mean"),
                mean_cpu_system=("cpu_system_sec", "mean"),
                mean_io_read_mb=("io_read_mb", "mean"),
                mean_io_write_mb=("io_write_mb", "mean"),
            )
            .round(3)
        )

        print("Shell Command Statistics (across all shards):")
        display(tool_stats.sort_values("total_elapsed", ascending=False))

        # Average shell overhead per shard
        per_shard = shell_end.groupby(["run_id", "task_id"]).agg(
            n_commands=("elapsed_sec", "count"),
            total_shell_sec=("elapsed_sec", "sum"),
            total_cpu_user=("cpu_user_sec", "sum"),
            total_cpu_system=("cpu_system_sec", "sum"),
        )
        print("\nPer-shard shell overhead stats:")
        per_shard.describe().round(2)
else:
    print("No _run_shell_command events found")

In [ ]:
# ── 4f: annotate_vcf_with_xgb_proba sub-stage breakdown ──────────────────────
# Shell commands within annotation: bgzip, tabix, bcftools annotate
# Plus the TSV write time (annotate_vcf_with_xgb_proba:exit - sum of shell commands)

sub_stages_annot = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    row = {"run_id": run_id, "task_id": task_id}

    annot_exit = group[(group["stage"] == "annotate_vcf_with_xgb_proba") & (group["phase"] == "exit")]
    if annot_exit.empty:
        continue
    row["total_sec"] = annot_exit["elapsed_sec"].iloc[0]

    # Shell commands within this stage: get the commands between annotate entry and exit
    shell_end = group[(group["stage"] == "_run_shell_command") & (group["phase"] == "end")]
    if not shell_end.empty and "cmd" in shell_end.columns:
        for _, cmd_row in shell_end.iterrows():
            cmd = str(cmd_row.get("cmd", ""))
            if "bgzip" in cmd:
                row["bgzip_sec"] = cmd_row["elapsed_sec"]
            elif "tabix" in cmd:
                row["tabix_sec"] = cmd_row["elapsed_sec"]

    # bcftools annotate (from vcf_utils)
    annot_vcf_exit = group[(group["stage"] == "annotate_vcf") & (group["phase"] == "exit")]
    if not annot_vcf_exit.empty:
        row["bcftools_annotate_sec"] = annot_vcf_exit["elapsed_sec"].iloc[0]
    else:
        row["bcftools_annotate_sec"] = 0

    row.setdefault("bgzip_sec", 0)
    row.setdefault("tabix_sec", 0)
    row["tsv_write_sec"] = max(0, row["total_sec"] - row["bgzip_sec"] - row["tabix_sec"] - row["bcftools_annotate_sec"])
    sub_stages_annot.append(row)

sub_annot_df = pd.DataFrame(sub_stages_annot)
if not sub_annot_df.empty:
    sub_cols = ["tsv_write_sec", "bgzip_sec", "tabix_sec", "bcftools_annotate_sec"]
    avg_sub = sub_annot_df[sub_cols].mean()

    fig_sub_annot = px.pie(
        names=["TSV write (Polars)", "bgzip", "tabix index", "bcftools annotate"],
        values=avg_sub.values,
        title="annotate_vcf_with_xgb_proba: Average Sub-Stage Breakdown",
        hole=0.35,
    )
    fig_sub_annot.update_traces(
        textinfo="label+percent+value", texttemplate="%{label}<br>%{percent:.1%}<br>%{value:.1f}s"
    )
    fig_sub_annot.show()

    print("\nannotate_vcf sub-stage stats (seconds):")
    sub_annot_df[["total_sec"] + sub_cols].describe().round(2)

In [ ]:
# ── 4g: Scatter plot - elapsed time vs. input size per stage ──────────────────
# Merge input sizes from entry events
entry_sizes = perf_df[perf_df["phase"] == "entry"][
    ["run_id", "task_id", "stage", "input_size_mb", "vcf_size_mb"]
].copy()
entry_sizes["size_mb"] = entry_sizes["input_size_mb"].fillna(entry_sizes["vcf_size_mb"])

exit_times = perf_df[perf_df["phase"] == "exit"][
    ["run_id", "task_id", "stage", "elapsed_sec", "variant_count", "row_count"]
].copy()
exit_times["count"] = exit_times["variant_count"].fillna(exit_times["row_count"])

scaling_df = exit_times.merge(
    entry_sizes[["run_id", "task_id", "stage", "size_mb"]],
    on=["run_id", "task_id", "stage"],
    how="left",
)
scaling_df = scaling_df[scaling_df["stage"].isin(MAIN_STAGES)]

fig_scatter = make_subplots(rows=2, cols=2, subplot_titles=MAIN_STAGES)
for i, stage in enumerate(MAIN_STAGES):
    sdf = scaling_df[scaling_df["stage"] == stage]
    row, col = (i // 2) + 1, (i % 2) + 1
    min_points_for_plot = 2
    if sdf["size_mb"].notna().sum() > min_points_for_plot:
        fig_scatter.add_trace(
            go.Scatter(x=sdf["size_mb"], y=sdf["elapsed_sec"], mode="markers", name=stage, showlegend=False),
            row=row,
            col=col,
        )
        fig_scatter.update_xaxes(title_text="Input Size (MB)", row=row, col=col)
    elif sdf["count"].notna().sum() > min_points_for_plot:
        fig_scatter.add_trace(
            go.Scatter(x=sdf["count"], y=sdf["elapsed_sec"], mode="markers", name=stage, showlegend=False),
            row=row,
            col=col,
        )
        fig_scatter.update_xaxes(title_text="Variant Count", row=row, col=col)
    fig_scatter.update_yaxes(title_text="Elapsed (sec)", row=row, col=col)

fig_scatter.update_layout(title="Scaling: Elapsed Time vs. Input Size per Stage", height=700)
fig_scatter.show()

## Section 5: Resource Efficiency Analysis

Quantify CPU, memory, and I/O efficiency per stage using `[PERF]` metrics.
- **CPU efficiency**: `cpu_user_sec / elapsed_sec` -- values <<1 indicate idle/waiting; values >1 indicate multi-threaded
- **System CPU ratio**: high `cpu_system_sec` relative to `cpu_user_sec` indicates subprocess/fork overhead
- **Memory growth**: cumulative `mem_delta_mb` across stages
- **I/O volumes**: per-stage and per-command read/write volumes

In [ ]:
# ── 5a: CPU efficiency per stage ──────────────────────────────────────────────
# For each task, compute cpu_user change between stage entry and exit via resource snapshots
cpu_efficiency_rows = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    for stage in MAIN_STAGES:
        entry = group[(group["stage"] == stage) & (group["phase"] == "entry")]
        exit_ = group[(group["stage"] == stage) & (group["phase"] == "exit")]
        if entry.empty or exit_.empty:
            continue

        elapsed = exit_["elapsed_sec"].iloc[0]
        if elapsed <= 0:
            continue

        # Resource snapshots from entry and exit
        cpu_user_start = entry["res_cpu_user_sec"].iloc[0] if "res_cpu_user_sec" in entry.columns else None
        cpu_user_end = exit_["res_cpu_user_sec"].iloc[0] if "res_cpu_user_sec" in exit_.columns else None
        cpu_sys_start = entry["res_cpu_system_sec"].iloc[0] if "res_cpu_system_sec" in entry.columns else None
        cpu_sys_end = exit_["res_cpu_system_sec"].iloc[0] if "res_cpu_system_sec" in exit_.columns else None

        if cpu_user_start is not None and cpu_user_end is not None:
            cpu_user_delta = cpu_user_end - cpu_user_start
            cpu_sys_delta = (
                (cpu_sys_end - cpu_sys_start) if (cpu_sys_start is not None and cpu_sys_end is not None) else 0
            )
        else:
            continue

        cpu_efficiency_rows.append(
            {
                "run_id": run_id,
                "task_id": task_id,
                "stage": stage,
                "elapsed_sec": elapsed,
                "cpu_user_delta": cpu_user_delta,
                "cpu_system_delta": cpu_sys_delta,
                "cpu_total_delta": cpu_user_delta + cpu_sys_delta,
                "cpu_efficiency": cpu_user_delta / elapsed,
                "cpu_system_ratio": cpu_sys_delta / (cpu_user_delta + cpu_sys_delta)
                if (cpu_user_delta + cpu_sys_delta) > 0
                else 0,
            }
        )

cpu_eff_df = pd.DataFrame(cpu_efficiency_rows)

if not cpu_eff_df.empty:
    fig_cpu = px.box(
        cpu_eff_df,
        x="stage",
        y="cpu_efficiency",
        color="stage",
        title="CPU Efficiency per Stage (cpu_user_sec / elapsed_sec)",
        labels={"cpu_efficiency": "CPU Efficiency (>1 = multi-threaded)", "stage": "Stage"},
        points="all",
    )
    fig_cpu.add_hline(y=1.0, line_dash="dash", line_color="red", annotation_text="1 core fully used")
    fig_cpu.update_layout(showlegend=False, xaxis_tickangle=-25)
    fig_cpu.show()

    print("\nCPU Efficiency Statistics:")
    display(cpu_eff_df.groupby("stage")[["cpu_efficiency", "cpu_system_ratio"]].describe().round(3))

    # Dynamic insight summary
    CPU_EFF_LOW = 0.1
    insights = []
    for stage in MAIN_STAGES:
        sdf = cpu_eff_df[cpu_eff_df["stage"] == stage]
        if sdf.empty:
            continue
        eff = sdf["cpu_efficiency"].mean()
        if eff < CPU_EFF_LOW:
            insights.append(
                f"- **{stage}**: CPU efficiency = {eff:.3f} — nearly zero user CPU in the Python process. "
                "This stage is dominated by subprocess execution (bcftools, bedtools, awk); "
                "the parent process is idle, waiting for child processes to finish."
            )
        elif eff < 1.0:
            insights.append(
                f"- **{stage}**: CPU efficiency = {eff:.2f} — partially CPU-active, "
                "but spends significant time waiting (I/O-bound or subprocess-bound)."
            )
        else:
            insights.append(
                f"- **{stage}**: CPU efficiency = {eff:.2f} — effectively multi-threaded, "
                "using more than 1 core on average."
            )

    print("\n=== CPU Efficiency Insights ===")
    print("CPU efficiency = cpu_user_sec / elapsed_sec. Values < 1 mean the Python process is idle/waiting.")
    print("Values > 1 indicate multi-threaded CPU work within the process.\n")
    for line in insights:
        print(line)

In [ ]:
# ── 5b: System CPU ratio (subprocess overhead indicator) ──────────────────────
if not cpu_eff_df.empty:
    fig_sys = px.box(
        cpu_eff_df,
        x="stage",
        y="cpu_system_ratio",
        color="stage",
        title="System CPU Ratio per Stage (high = subprocess/fork overhead)",
        labels={"cpu_system_ratio": "system / (user + system)", "stage": "Stage"},
        points="all",
    )
    fig_sys.update_layout(showlegend=False, xaxis_tickangle=-25)
    fig_sys.show()

    # Dynamic insight summary
    SYS_RATIO_HIGH = 0.4
    SYS_RATIO_MODERATE = 0.15
    insights = []
    for stage in MAIN_STAGES:
        sdf = cpu_eff_df[cpu_eff_df["stage"] == stage]
        if sdf.empty:
            continue
        sys_ratio = sdf["cpu_system_ratio"].mean()
        if sys_ratio > SYS_RATIO_HIGH:
            insights.append(
                f"- **{stage}**: system ratio = {sys_ratio:.2f} — high kernel overhead, "
                "suggesting frequent subprocess spawning, pipe I/O, or fork() calls."
            )
        elif sys_ratio > SYS_RATIO_MODERATE:
            insights.append(
                f"- **{stage}**: system ratio = {sys_ratio:.2f} — moderate kernel overhead, "
                "likely from some subprocess management or file operations."
            )
        else:
            insights.append(
                f"- **{stage}**: system ratio = {sys_ratio:.2f} — low kernel overhead, "
                "most CPU time is in user-space computation."
            )

    print("\n=== System CPU Ratio Insights ===")
    print("System CPU ratio = cpu_system / (cpu_user + cpu_system). High values indicate")
    print("the process spends CPU time in kernel calls (fork, pipe I/O, context switches).\n")
    for line in insights:
        print(line)

In [ ]:
# ── 5c: Memory growth waterfall across stages ────────────────────────────────
mem_rows = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    for stage in MAIN_STAGES:
        entry = group[(group["stage"] == stage) & (group["phase"] == "entry")]
        exit_ = group[(group["stage"] == stage) & (group["phase"] == "exit")]
        if entry.empty or exit_.empty:
            continue

        mem_start = (
            entry["mem_mb"].iloc[0] if "mem_mb" in entry.columns else entry.get("res_rss_mb", pd.Series([None])).iloc[0]
        )
        mem_end = (
            exit_["mem_mb"].iloc[0] if "mem_mb" in exit_.columns else exit_.get("res_rss_mb", pd.Series([None])).iloc[0]
        )
        mem_delta = exit_["mem_delta_mb"].iloc[0] if "mem_delta_mb" in exit_.columns else None

        mem_rows.append(
            {
                "run_id": run_id,
                "task_id": task_id,
                "stage": stage,
                "mem_start_mb": mem_start,
                "mem_end_mb": mem_end,
                "mem_delta_mb": mem_delta
                if mem_delta is not None
                else (mem_end - mem_start if mem_start and mem_end else None),
            }
        )

mem_df = pd.DataFrame(mem_rows)

if not mem_df.empty:
    # Waterfall: average memory at entry and exit of each stage
    mem_summary = mem_df.groupby("stage")[["mem_start_mb", "mem_end_mb", "mem_delta_mb"]].mean().reindex(MAIN_STAGES)

    fig_mem = go.Figure()
    fig_mem.add_trace(
        go.Bar(name="Memory at Entry", x=MAIN_STAGES, y=mem_summary["mem_start_mb"], marker_color="lightblue")
    )
    fig_mem.add_trace(
        go.Bar(name="Memory at Exit", x=MAIN_STAGES, y=mem_summary["mem_end_mb"], marker_color="steelblue")
    )
    fig_mem.update_layout(
        title="Average Memory Usage at Stage Entry vs Exit",
        yaxis_title="RSS (MB)",
        barmode="group",
        xaxis_tickangle=-25,
    )
    fig_mem.show()

    # Memory delta box plot
    fig_mem_delta = px.box(
        mem_df,
        x="stage",
        y="mem_delta_mb",
        color="stage",
        title="Memory Growth per Stage (mem_delta_mb)",
        labels={"mem_delta_mb": "Memory Delta (MB)", "stage": "Stage"},
        points="all",
    )
    fig_mem_delta.update_layout(showlegend=False, xaxis_tickangle=-25)
    fig_mem_delta.show()

    print("\nMemory statistics per stage (MB):")
    display(mem_df.groupby("stage")[["mem_start_mb", "mem_end_mb", "mem_delta_mb"]].describe().round(1))

    # Dynamic insight summary
    MEM_DELTA_NEGLIGIBLE = 5
    MEM_DELTA_SIGNIFICANT = 100

    print("\n=== Memory Growth Insights ===")
    print("Memory delta shows how much RSS grows during each stage.\n")
    for stage in MAIN_STAGES:
        sdf = mem_df[mem_df["stage"] == stage]
        if sdf.empty:
            continue
        delta = sdf["mem_delta_mb"].mean()
        end_mb = sdf["mem_end_mb"].mean()
        if pd.isna(delta):
            continue
        if abs(delta) < MEM_DELTA_NEGLIGIBLE:
            print(
                f"- **{stage}**: avg delta = {delta:.1f} MB — negligible memory change. "
                "Stage operates in constant memory (subprocess-based work doesn't grow Python RSS)."
            )
        elif delta > MEM_DELTA_SIGNIFICANT:
            print(
                f"- **{stage}**: avg delta = +{delta:.0f} MB (ends at {end_mb:.0f} MB) — "
                "significant memory growth, likely from loading data into DataFrames/arrays."
            )
        else:
            print(f"- **{stage}**: avg delta = {delta:+.1f} MB — " f"moderate memory change (ends at {end_mb:.0f} MB).")

In [ ]:
# ── 5d: I/O volume analysis per stage ─────────────────────────────────────────
io_rows = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    for stage in MAIN_STAGES:
        entry = group[(group["stage"] == stage) & (group["phase"] == "entry")]
        exit_ = group[(group["stage"] == stage) & (group["phase"] == "exit")]
        if entry.empty or exit_.empty:
            continue

        io_read_start = entry["res_io_read_mb"].iloc[0] if "res_io_read_mb" in entry.columns else None
        io_read_end = exit_["res_io_read_mb"].iloc[0] if "res_io_read_mb" in exit_.columns else None
        io_write_start = entry["res_io_write_mb"].iloc[0] if "res_io_write_mb" in entry.columns else None
        io_write_end = exit_["res_io_write_mb"].iloc[0] if "res_io_write_mb" in exit_.columns else None

        io_rows.append(
            {
                "run_id": run_id,
                "task_id": task_id,
                "stage": stage,
                "io_read_mb": (io_read_end - io_read_start)
                if (io_read_start is not None and io_read_end is not None)
                else None,
                "io_write_mb": (io_write_end - io_write_start)
                if (io_write_start is not None and io_write_end is not None)
                else None,
            }
        )

io_df = pd.DataFrame(io_rows)

if not io_df.empty and io_df["io_read_mb"].notna().sum() > 0:
    io_summary = io_df.groupby("stage")[["io_read_mb", "io_write_mb"]].mean().reindex(MAIN_STAGES)

    fig_io = go.Figure()
    fig_io.add_trace(
        go.Bar(
            name="Read (MB)",
            x=MAIN_STAGES,
            y=io_summary["io_read_mb"],
            marker_color="cornflowerblue",
        )
    )
    fig_io.add_trace(
        go.Bar(
            name="Write (MB)",
            x=MAIN_STAGES,
            y=io_summary["io_write_mb"],
            marker_color="salmon",
        )
    )
    fig_io.update_layout(
        title="Average I/O Volume per Stage",
        yaxis_title="MB",
        barmode="group",
        xaxis_tickangle=-25,
    )
    fig_io.show()

    # I/O amplification: total written vs final output size
    pipeline_exits = perf_df[(perf_df["stage"] == "somatic_featuremap_classifier") & (perf_df["phase"] == "exit")]
    if not pipeline_exits.empty and "output_size_mb" in pipeline_exits.columns:
        avg_output = pipeline_exits["output_size_mb"].mean()
        total_write = io_summary["io_write_mb"].sum()
        ratio = total_write / avg_output
        print(
            f"I/O Amplification: avg total write = {total_write:.1f} MB, "
            f"avg output = {avg_output:.1f} MB, ratio = {ratio:.1f}x"
        )

    print("\nI/O volume per stage (MB):")
    display(io_df.groupby("stage")[["io_read_mb", "io_write_mb"]].describe().round(1))

    # Dynamic insight summary
    print("\n=== I/O Volume Insights ===")
    io_summary_local = io_df.groupby("stage")[["io_read_mb", "io_write_mb"]].mean().reindex(MAIN_STAGES)
    total_read = io_summary_local["io_read_mb"].sum()
    total_write = io_summary_local["io_write_mb"].sum()
    print(f"Total average I/O per shard: {total_read:.0f} MB read, {total_write:.0f} MB write\n")
    for stage in MAIN_STAGES:
        if stage not in io_summary_local.index:
            continue
        r = io_summary_local.loc[stage, "io_read_mb"]
        w = io_summary_local.loc[stage, "io_write_mb"]
        if pd.isna(r):
            continue
        read_pct = r / total_read * 100 if total_read > 0 else 0
        write_pct = w / total_write * 100 if total_write > 0 else 0
        print(f"- **{stage}**: read {r:.0f} MB ({read_pct:.0f}%), write {w:.0f} MB ({write_pct:.0f}%)")
else:
    print("No I/O counters available in [PERF] logs")

## Section 6: Architectural Bottleneck Identification

Map performance data to specific architectural decisions:
- **awk overhead**: time in bcftools+awk pipelines vs. total vcf_to_parquet time
- **Subprocess spawn cost**: number and cumulative cost of subprocess invocations
- **Format conversion cost**: Polars-to-Pandas and TSV round-trips
- **I/O amplification**: intermediate files vs. final output
- **Parallelism efficiency**: ProcessPoolExecutor overhead

In [ ]:
# ── 6a: Comprehensive bottleneck summary table ───────────────────────────────
from IPython.display import Markdown
from IPython.display import display as ipy_display

bottleneck_data = {}

# 1. Stage time shares
if not timing_pivot.empty:
    total_mean = timing_pivot["total_sec"].mean()
    for stage in MAIN_STAGES:
        pct = timing_pivot[stage].mean() / total_mean * 100
        bottleneck_data[f"stage_{stage}_pct"] = pct
        bottleneck_data[f"stage_{stage}_mean_sec"] = timing_pivot[stage].mean()
    bottleneck_data["total_pipeline_mean_sec"] = total_mean

# 2. bcftools+awk (region_jobs) share of vcf_to_parquet
if not sub_vtp_df.empty:
    rj_pct = sub_vtp_df["region_jobs_sec"].mean() / sub_vtp_df["total_sec"].mean() * 100
    bottleneck_data["region_jobs_pct_of_vtp"] = rj_pct
    bottleneck_data["region_jobs_mean_sec"] = sub_vtp_df["region_jobs_sec"].mean()
    bottleneck_data["merge_mean_sec"] = sub_vtp_df["merge_sec"].mean()

# 3. Polars-to-Pandas conversion overhead
if not sub_clf_df.empty:
    pandas_pct = sub_clf_df["pandas_convert_sec"].mean() / sub_clf_df["total_sec"].mean() * 100
    bottleneck_data["pandas_convert_pct_of_classifier"] = pandas_pct
    bottleneck_data["pandas_convert_mean_sec"] = sub_clf_df["pandas_convert_sec"].mean()
    if "mem_delta_mb" in sub_clf_df.columns:
        bottleneck_data["classifier_mem_delta_mean_mb"] = sub_clf_df["mem_delta_mb"].mean()

# 4. Subprocess overhead
if not shell_cmds.empty:
    shell_end_all = shell_cmds[shell_cmds["phase"] == "end"]
    per_shard_shells = shell_end_all.groupby(["run_id", "task_id"]).agg(
        n_commands=("elapsed_sec", "count"),
        total_sec=("elapsed_sec", "sum"),
    )
    bottleneck_data["avg_shell_commands_per_shard"] = per_shard_shells["n_commands"].mean()
    bottleneck_data["avg_shell_total_sec_per_shard"] = per_shard_shells["total_sec"].mean()
    if total_mean > 0:
        bottleneck_data["shell_pct_of_pipeline"] = per_shard_shells["total_sec"].mean() / total_mean * 100

# 5. TSV round-trip overhead (annotation)
if not sub_annot_df.empty:
    tsv_overhead = sub_annot_df[["tsv_write_sec", "bgzip_sec", "tabix_sec"]].sum(axis=1).mean()
    bottleneck_data["tsv_roundtrip_mean_sec"] = tsv_overhead
    bottleneck_data["tsv_roundtrip_pct_of_annotation"] = tsv_overhead / sub_annot_df["total_sec"].mean() * 100

# 6. CPU efficiency
if not cpu_eff_df.empty:
    for stage in MAIN_STAGES:
        sdf = cpu_eff_df[cpu_eff_df["stage"] == stage]
        if not sdf.empty:
            bottleneck_data[f"cpu_eff_{stage}"] = sdf["cpu_efficiency"].mean()
            bottleneck_data[f"cpu_sys_ratio_{stage}"] = sdf["cpu_system_ratio"].mean()

bottleneck_summary = pd.Series(bottleneck_data).round(3)

# Structured markdown report instead of raw key-value dump
CPU_EFF_THRESHOLD = 0.1
REGION_JOBS_DOMINANT = 95

total_sec = bottleneck_data.get("total_pipeline_mean_sec", 0)

report_lines = [
    "## Architectural Bottleneck Summary",
    "",
    f"**Average pipeline wall-clock time per shard: {total_sec:.1f}s**",
    "",
    "### 1. Stage Time Shares",
    "",
    "| Stage | Mean (s) | Share (%) | CPU Efficiency | System CPU Ratio |",
    "|-------|----------|-----------|----------------|------------------|",
]

for stage in MAIN_STAGES:
    mean_s = bottleneck_data.get(f"stage_{stage}_mean_sec", 0)
    pct = bottleneck_data.get(f"stage_{stage}_pct", 0)
    cpu_e = bottleneck_data.get(f"cpu_eff_{stage}", float("nan"))
    sys_r = bottleneck_data.get(f"cpu_sys_ratio_{stage}", float("nan"))
    short = stage.replace("_", " ")
    report_lines.append(f"| {short} | {mean_s:.1f} | {pct:.1f}% | {cpu_e:.3f} | {sys_r:.3f} |")

report_lines += [
    "",
    "### 2. Dominant Bottleneck: VCF-to-Parquet (bcftools+awk)",
    "",
    f"- Region jobs (bcftools+awk parallel processing): **{bottleneck_data.get('region_jobs_pct_of_vtp', 0):.1f}%** "
    f"of vcf_to_parquet time (avg **{bottleneck_data.get('region_jobs_mean_sec', 0):.1f}s**)",
    f"- Parquet merge: avg **{bottleneck_data.get('merge_mean_sec', 0):.2f}s** (negligible)",
    "",
    "### 3. Subprocess Overhead",
    "",
    f"- Shell commands per shard: **{bottleneck_data.get('avg_shell_commands_per_shard', 0):.0f}**",
    f"- Total shell time per shard: **{bottleneck_data.get('avg_shell_total_sec_per_shard', 0):.2f}s** "
    f"(**{bottleneck_data.get('shell_pct_of_pipeline', 0):.1f}%** of pipeline)",
    "- Note: this only counts `_run_shell_command` events — the larger subprocess cost is in "
    "`bcftools query | awk` within ProcessPoolExecutor workers (captured under `region_jobs` time)",
    "",
    "### 4. Format Conversion Overhead",
    "",
    f"- Polars→Pandas conversion: **{bottleneck_data.get('pandas_convert_mean_sec', 0):.2f}s** "
    f"(**{bottleneck_data.get('pandas_convert_pct_of_classifier', 0):.1f}%** of classifier stage)",
    f"- Classifier memory growth: **{bottleneck_data.get('classifier_mem_delta_mean_mb', 0):.0f} MB** "
    "(from DataFrame duplication)",
    f"- TSV round-trip (write+bgzip+tabix): **{bottleneck_data.get('tsv_roundtrip_mean_sec', 0):.2f}s** "
    f"(**{bottleneck_data.get('tsv_roundtrip_pct_of_annotation', 0):.1f}%** of annotation stage)",
    "",
    "### 5. Key Observations",
    "",
]

# Dynamic observations
cpu_eff_filter = bottleneck_data.get("cpu_eff_filter_and_annotate_tr", 0)
cpu_eff_read = bottleneck_data.get("cpu_eff_read_vcf_with_aggregation", 0)
if cpu_eff_filter < CPU_EFF_THRESHOLD and cpu_eff_read < CPU_EFF_THRESHOLD:
    report_lines.append(
        "- **Both dominant stages** (`filter_and_annotate_tr` and `read_vcf_with_aggregation`) "
        f"have near-zero CPU efficiency ({cpu_eff_filter:.3f} and {cpu_eff_read:.3f}). "
        "The Python process is idle while subprocess children (bcftools, bedtools, awk) do all the work."
    )

filter_pct = bottleneck_data.get("stage_filter_and_annotate_tr_pct", 0)
read_pct = bottleneck_data.get("stage_read_vcf_with_aggregation_pct", 0)
report_lines.append(
    f"- `filter_and_annotate_tr` ({filter_pct:.0f}%) and `read_vcf_with_aggregation` ({read_pct:.0f}%) "
    f"together consume **{filter_pct + read_pct:.0f}%** of pipeline time."
)

rj_pct = bottleneck_data.get("region_jobs_pct_of_vtp", 0)
if rj_pct > REGION_JOBS_DOMINANT:
    report_lines.append(
        f"- **{rj_pct:.0f}%** of vcf_to_parquet time is in `region_jobs` (bcftools query | awk) — "
        "replacing awk with Polars native operations and/or replacing bcftools with cyvcf2 "
        "would eliminate this overhead."
    )

ipy_display(Markdown("\n".join(report_lines)))

In [ ]:
# ── 6b: Visualize bottleneck decomposition (sunburst chart) ──────────────────
# Build hierarchical data: Pipeline -> Stage -> Sub-Stage
sunburst_data = []

# Stage-level
for stage in MAIN_STAGES:
    mean_sec = timing_pivot[stage].mean() if stage in timing_pivot.columns else 0
    sunburst_data.append({"parent": "Pipeline", "name": stage, "value": mean_sec})

# Sub-stages for filter_and_annotate_tr
if not sub_filter_df.empty:
    for sub, label in [
        ("view_vcf_sec", "bcftools view + index"),
        ("bedtools_pipeline_sec", "bedtools closest pipeline"),
        ("annotate_vcf_sec", "bcftools annotate + index"),
    ]:
        sunburst_data.append({"parent": "filter_and_annotate_tr", "name": label, "value": sub_filter_df[sub].mean()})

# Sub-stages for read_vcf_with_aggregation
if not sub_read_vcf_df.empty:
    for sub, label in [
        ("vcf_to_parquet_sec", "vcf_to_parquet"),
        ("pileup_sec", "pileup_features"),
        ("postproc_sec", "post_processing"),
        ("other_sec", "other"),
    ]:
        sunburst_data.append(
            {"parent": "read_vcf_with_aggregation", "name": label, "value": sub_read_vcf_df[sub].mean()}
        )

# Sub-stages for vcf_to_parquet
if not sub_vtp_df.empty:
    for sub, label in [
        ("region_jobs_sec", "bcftools+awk (parallel)"),
        ("merge_sec", "parquet_merge"),
        ("other_sec", "setup/teardown"),
    ]:
        sunburst_data.append({"parent": "vcf_to_parquet", "name": label, "value": sub_vtp_df[sub].mean()})

# Sub-stages for run_classifier
if not sub_clf_df.empty:
    for sub, label in [
        ("pandas_convert_sec", "Polars→Pandas"),
        ("model_load_sec", "model_load"),
        ("predict_sec", "XGBoost predict"),
        ("other_sec", "other"),
    ]:
        sunburst_data.append({"parent": "run_classifier", "name": label, "value": sub_clf_df[sub].mean()})

# Sub-stages for annotate_vcf_with_xgb_proba
if not sub_annot_df.empty:
    for sub, label in [
        ("tsv_write_sec", "TSV write (Polars)"),
        ("bgzip_sec", "bgzip"),
        ("tabix_sec", "tabix index"),
        ("bcftools_annotate_sec", "bcftools annotate"),
    ]:
        sunburst_data.append(
            {"parent": "annotate_vcf_with_xgb_proba", "name": label, "value": sub_annot_df[sub].mean()}
        )

sb_df = pd.DataFrame(sunburst_data)
sb_df["value"] = sb_df["value"].clip(lower=0)

fig_sun = px.sunburst(
    sb_df,
    names="name",
    parents="parent",
    values="value",
    title="Pipeline Time Decomposition (avg seconds)",
    branchvalues="total",
)
fig_sun.update_traces(textinfo="label+value+percent parent")
fig_sun.update_layout(height=600)
fig_sun.show()

In [ ]:
# ── 6c: Parallelism efficiency analysis ──────────────────────────────────────
# Measure actual CPU utilization during parallel region processing by comparing
# process-level CPU time delta (from resource snapshots) to wall-clock time.
# Since workers are subprocess-based (bcftools+awk), parent-process CPU will be low,
# so we also compute throughput metrics.

if not sub_vtp_df.empty and "region_count" in sub_vtp_df.columns:
    sub_vtp_with_jobs = sub_vtp_df.dropna(subset=["region_count"]).copy()

    # Compute throughput: regions per second and MB per second
    sub_vtp_with_jobs["regions_per_sec"] = sub_vtp_with_jobs["region_count"] / sub_vtp_with_jobs["region_jobs_sec"]

    # Get CPU time delta during region processing from resource snapshots
    parallelism_rows = []
    for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
        rj_entry = group[(group["stage"] == "_run_region_jobs") & (group["phase"] == "entry")]
        rj_exit = group[(group["stage"] == "_run_region_jobs") & (group["phase"] == "exit")]
        if rj_entry.empty or rj_exit.empty:
            continue
        elapsed = rj_exit["elapsed_sec"].iloc[0]
        if elapsed <= 0:
            continue
        cpu_start = rj_entry["res_cpu_user_sec"].iloc[0] if "res_cpu_user_sec" in rj_entry.columns else None
        cpu_end = rj_exit["res_cpu_user_sec"].iloc[0] if "res_cpu_user_sec" in rj_exit.columns else None
        io_start = rj_entry["res_io_read_mb"].iloc[0] if "res_io_read_mb" in rj_entry.columns else None
        io_end = rj_exit["res_io_read_mb"].iloc[0] if "res_io_read_mb" in rj_exit.columns else None
        parallelism_rows.append(
            {
                "run_id": run_id,
                "task_id": task_id,
                "elapsed_sec": elapsed,
                "parent_cpu_sec": (cpu_end - cpu_start) if (cpu_start is not None and cpu_end is not None) else None,
                "io_read_mb": (io_end - io_start) if (io_start is not None and io_end is not None) else None,
            }
        )

    par_df = pd.DataFrame(parallelism_rows)
    if not par_df.empty and par_df["parent_cpu_sec"].notna().sum() > 0:
        par_df["parent_cpu_ratio"] = par_df["parent_cpu_sec"] / par_df["elapsed_sec"]
        par_df["io_throughput_mb_per_sec"] = par_df["io_read_mb"] / par_df["elapsed_sec"]

    if "jobs" in sub_vtp_with_jobs.columns:
        sub_vtp_with_jobs = sub_vtp_with_jobs.dropna(subset=["jobs"])
        avg_jobs = sub_vtp_with_jobs["jobs"].mean()
        avg_regions = sub_vtp_with_jobs["region_count"].mean()
        avg_throughput = sub_vtp_with_jobs["regions_per_sec"].mean()
    else:
        avg_jobs = None
        avg_regions = sub_vtp_with_jobs["region_count"].mean()
        avg_throughput = sub_vtp_with_jobs["regions_per_sec"].mean()

    # Throughput distribution
    fig_par = px.histogram(
        sub_vtp_with_jobs,
        x="regions_per_sec",
        nbins=30,
        title="Region Processing Throughput Distribution",
        labels={"regions_per_sec": "Regions / second"},
    )
    fig_par.show()

    # Summary
    print("=== Parallelism Analysis ===\n")
    if avg_jobs is not None:
        print(f"  Workers (jobs): {avg_jobs:.0f}")
    print(f"  Average regions per shard: {avg_regions:.0f}")
    print(f"  Average throughput: {avg_throughput:.1f} regions/sec")
    print(f"  Average region_jobs wall time: {sub_vtp_with_jobs['region_jobs_sec'].mean():.1f}s")
    if not par_df.empty and par_df["parent_cpu_ratio"].notna().sum() > 0:
        print(f"\n  Parent process CPU ratio during region jobs: " f"{par_df['parent_cpu_ratio'].mean():.3f}")
        print("  (This is near-zero because actual work happens in subprocess children)")
        print(f"  I/O throughput: {par_df['io_throughput_mb_per_sec'].mean():.1f} MB/sec")
    print("\n  Note: ProcessPoolExecutor spawns subprocess-based workers (bcftools|awk).")
    print(f"  With only {avg_jobs:.0f} worker(s), parallelism gain is limited. The bottleneck is")
    print("  the sequential bcftools+awk pipeline per region, not the Python process pool.")

In [ ]:
# ── 6d: File size analysis - intermediate vs final ───────────────────────────
# Track file sizes through the pipeline to identify I/O amplification from intermediate files
file_size_rows = []
for (run_id, task_id), group in perf_df.groupby(["run_id", "task_id"]):
    row = {"run_id": run_id, "task_id": task_id}

    # Input VCF size
    pipeline_entry = group[(group["stage"] == "somatic_featuremap_classifier") & (group["phase"] == "entry")]
    if not pipeline_entry.empty and "input_size_mb" in pipeline_entry.columns:
        row["input_vcf_mb"] = pipeline_entry["input_size_mb"].iloc[0]

    # Filtered VCF (output of filter_and_annotate_tr)
    step1_exit = group[(group["stage"] == "filter_and_annotate_tr") & (group["phase"] == "exit")]
    if not step1_exit.empty and "output_size_mb" in step1_exit.columns:
        row["filtered_vcf_mb"] = step1_exit["output_size_mb"].iloc[0]

    # Parquet file size
    vtp_end = group[(group["stage"] == "vcf_to_parquet") & (group["phase"] == "end")]
    if not vtp_end.empty and "parquet_size_mb" in vtp_end.columns:
        row["parquet_mb"] = vtp_end["parquet_size_mb"].iloc[0]

    # Output VCF
    pipeline_exit = group[(group["stage"] == "somatic_featuremap_classifier") & (group["phase"] == "exit")]
    if not pipeline_exit.empty and "output_size_mb" in pipeline_exit.columns:
        row["output_vcf_mb"] = pipeline_exit["output_size_mb"].iloc[0]

    file_size_rows.append(row)

file_size_df = pd.DataFrame(file_size_rows)
size_cols = [c for c in ["input_vcf_mb", "filtered_vcf_mb", "parquet_mb", "output_vcf_mb"] if c in file_size_df.columns]

if size_cols:
    print("File Size Flow (MB) - mean across all shards:")
    print(file_size_df[size_cols].mean().round(2).to_string())

    min_cols_for_chart = 2
    if len(size_cols) >= min_cols_for_chart:
        avg_sizes = file_size_df[size_cols].mean()
        fig_sizes = go.Figure(
            go.Bar(
                x=[c.replace("_mb", "") for c in size_cols],
                y=avg_sizes.to_numpy(),
                text=[f"{v:.1f} MB" for v in avg_sizes.to_numpy()],
                textposition="auto",
            )
        )
        fig_sizes.update_layout(
            title="Average File Sizes Through Pipeline Stages",
            yaxis_title="Size (MB)",
            xaxis_title="Artifact",
        )
        fig_sizes.show()

## Section 7: Cross-Sample Scaling

How does performance scale across the 12 samples? Are certain samples disproportionately slow?

In [ ]:
# ── 7a: Per-run summary ──────────────────────────────────────────────────────
# Aggregate per run_id: total time, shard count, total variants, per-stage breakdown
pipeline_exits = perf_df[(perf_df["stage"] == "somatic_featuremap_classifier") & (perf_df["phase"] == "exit")].copy()

run_summary_rows = []
for run_id, group in pipeline_exits.groupby("run_id"):
    row = {
        "run_id": run_id,
        "n_shards": len(group),
        "total_variants": group["variant_count"].sum() if "variant_count" in group.columns else None,
        "mean_shard_sec": group["elapsed_sec"].mean(),
        "max_shard_sec": group["elapsed_sec"].max(),
        "min_shard_sec": group["elapsed_sec"].min(),
        "sum_shard_sec": group["elapsed_sec"].sum(),
        "mean_input_mb": group.get("input_size_mb", pd.Series([None])).mean(),
        "mean_output_mb": group.get("output_size_mb", pd.Series([None])).mean(),
    }
    run_summary_rows.append(row)

run_summary = pd.DataFrame(run_summary_rows)
if not run_summary.empty:
    # Add per-stage averages
    for stage in MAIN_STAGES:
        stage_data = timing_pivot.groupby("run_id")[stage].mean()
        run_summary = run_summary.merge(
            stage_data.reset_index().rename(columns={stage: f"avg_{stage}_sec"}),
            on="run_id",
            how="left",
        )

    print("Per-Run Summary:")
    display(run_summary.round(2))
else:
    print("No pipeline exit events found")

In [ ]:
# ── 7b: Heatmap - stage timing across runs ───────────────────────────────────
if not run_summary.empty:
    stage_cols = [f"avg_{s}_sec" for s in MAIN_STAGES if f"avg_{s}_sec" in run_summary.columns]
    if stage_cols:
        heatmap_df = run_summary.set_index("run_id")[stage_cols]
        heatmap_df.columns = [c.replace("avg_", "").replace("_sec", "") for c in stage_cols]

        fig_heat = px.imshow(
            heatmap_df.values,
            x=heatmap_df.columns.tolist(),
            y=[rid[:8] for rid in heatmap_df.index],
            color_continuous_scale="YlOrRd",
            title="Stage Timing Heatmap Across Runs (avg seconds per shard)",
            labels={"x": "Stage", "y": "Run ID", "color": "Seconds"},
            text_auto=".1f",
            aspect="auto",
        )
        fig_heat.update_layout(height=max(400, len(run_summary) * 40))
        fig_heat.show()

In [ ]:
# ── 7c: Scaling analysis ─────────────────────────────────────────────────────
# Total time vs total variant count across runs
MIN_SAMPLES_FOR_TRENDLINE = 3
if not run_summary.empty and "total_variants" in run_summary.columns:
    rs = run_summary.dropna(subset=["total_variants"])
    if len(rs) >= MIN_SAMPLES_FOR_TRENDLINE:
        fig_scale = px.scatter(
            rs,
            x="total_variants",
            y="sum_shard_sec",
            text=[rid[:8] for rid in rs["run_id"]],
            trendline="ols",
            title="Total Compute Time vs. Variant Count (across runs)",
            labels={"total_variants": "Total Variants", "sum_shard_sec": "Sum of Shard Times (sec)"},
        )
        fig_scale.update_traces(textposition="top center")
        fig_scale.show()

        # Per-shard scaling
        fig_shard_scale = px.scatter(
            rs,
            x="total_variants",
            y="max_shard_sec",
            text=[rid[:8] for rid in rs["run_id"]],
            trendline="ols",
            title="Slowest Shard Time vs. Variant Count (across runs)",
            labels={"total_variants": "Total Variants", "max_shard_sec": "Max Shard Time (sec)"},
        )
        fig_shard_scale.update_traces(textposition="top center")
        fig_shard_scale.show()

## Section 8: Architectural Recommendations

Data-backed analysis of each architectural improvement opportunity.
The cell below auto-generates a report based on the bottleneck metrics computed above.

In [ ]:
# ── 8: Auto-generated recommendations report ────────────────────────────────
from IPython.display import Markdown
from IPython.display import display as ipy_display

total_sec = bottleneck_data.get("total_pipeline_mean_sec", 0)


def _pct(key, fallback=0):
    return bottleneck_data.get(key, fallback)


def _sec(key, fallback=0):
    return bottleneck_data.get(key, fallback)


recommendations = []

# ─── Recommendation 1: Replace awk with Polars native operations ─────────────
rj_sec = _sec("region_jobs_mean_sec")
rj_pct_vtp = _pct("region_jobs_pct_of_vtp")
recommendations.append(f"""
### 1. Replace `awk` scripts with Polars native list operations

**Current**: Each region-sample pair spawns `bcftools query | awk` as a subprocess pipeline.
The awk scripts (`explode_lists.awk`, `aggregate_lists.awk`) perform list field explosion
and aggregation (mean, min, max, count) as string processing on TSV output.

**Data**: Region jobs (bcftools+awk) account for **{rj_pct_vtp:.1f}%** of vcf_to_parquet time
(avg **{rj_sec:.1f}s** per shard).

**Proposed**: Replace awk with Polars native operations:
- `pl.col("field").str.split(",").list.eval(...)` or `explode()` for list explosion
- `pl.col("field").list.mean()`, `.list.max()`, `.list.min()`, `.list.len()` for aggregation
- This eliminates all subprocess overhead for awk and the TSV string intermediate

**Impact**: Removes N subprocess invocations per region (where N = number of samples).
Polars operates on columnar memory without serialization overhead.

**Complexity**: Medium -- requires rewriting `_bcftools_awk_stdout()` and the awk scripts
into Polars expressions, but Polars has all the necessary list operations built in.
""")

# ─── Recommendation 2: Replace bcftools query with cyvcf2 ────────────────────
shell_pct = _pct("shell_pct_of_pipeline")
avg_cmds = _sec("avg_shell_commands_per_shard")
cpu_eff_read = _pct("cpu_eff_read_vcf_with_aggregation", 0)
recommendations.append(f"""
### 2. Replace `bcftools query` with `cyvcf2` for VCF-to-DataFrame conversion

**Current**: VCF fields are extracted via `bcftools query -f` (subprocess), piped to awk,
producing TSV strings that are then parsed into Polars DataFrames.
Each shard runs ~{avg_cmds:.0f} shell commands, consuming **{shell_pct:.1f}%** of pipeline time.

**Data**: CPU efficiency during `read_vcf_with_aggregation` is **{cpu_eff_read:.2f}**
(values < 1.0 indicate the process is idle, waiting for subprocesses).

**Proposed**: Use `cyvcf2.VCF` (C-backed, htslib-based) to iterate over VCF records
and directly populate numpy arrays or Polars Series:
```python
import cyvcf2
vcf = cyvcf2.VCF(vcf_path)
for variant in vcf:
    # variant.INFO.get("FIELD"), variant.format("AD"), etc.
    # Directly build columnar arrays
```

**Impact**: Eliminates all bcftools+awk subprocess overhead. cyvcf2 is a thin C wrapper
around htslib and can parse VCFs as fast as bcftools. Combined with Recommendation 1,
this replaces the entire `bcftools query | awk | TSV string | Polars` chain with a single
`cyvcf2 | numpy arrays | Polars DataFrame` pass.

**Complexity**: Medium-High -- requires rewriting `_stream_region_to_polars()` and the
region processing logic in `featuremap_to_dataframe.py`.
""")

# ─── Recommendation 3: Eliminate Polars-to-Pandas conversion ─────────────────
pandas_sec = _sec("pandas_convert_mean_sec")
pandas_pct = _pct("pandas_convert_pct_of_classifier")
clf_mem = _sec("classifier_mem_delta_mean_mb")
recommendations.append(f"""
### 3. Eliminate Polars-to-Pandas conversion for XGBoost

**Current**: The entire Polars DataFrame is converted to Pandas (`df.to_pandas()`)
before being passed to XGBoost.

**Data**: Conversion takes **{pandas_sec:.2f}s** avg (**{pandas_pct:.1f}%** of classifier time).
Memory growth during the classifier stage averages **{clf_mem:.0f} MB**.

**Proposed**: XGBoost's `DMatrix` accepts numpy arrays directly:
```python
feature_matrix = df_variants.select(model_features).to_numpy()
dmatrix = xgb.DMatrix(feature_matrix, feature_names=model_features)
predictions = xgb_clf.get_booster().predict(dmatrix)
```

This avoids the full DataFrame copy that `to_pandas()` performs.

**Impact**: Saves {pandas_sec:.2f}s and ~{clf_mem:.0f} MB memory per shard.
Minor but easy win.

**Complexity**: Low -- a few lines of code change. The existing TODO in the code
already notes this: "don't convert to pandas, keep as polars dataframe".
""")

# ─── Recommendation 4: In-memory VCF annotation ─────────────────────────────
tsv_sec = _sec("tsv_roundtrip_mean_sec")
tsv_pct = _pct("tsv_roundtrip_pct_of_annotation")
annot_sec = _sec("stage_annotate_vcf_with_xgb_proba_mean_sec")
recommendations.append(f"""
### 4. In-memory VCF annotation via `pysam` instead of TSV round-trip

**Current**: The annotation pipeline is:
`Polars DataFrame → write TSV → bgzip → tabix index → bcftools annotate → output VCF`.
This creates 3 intermediate files and runs 3 subprocesses for a single INFO field.

**Data**: TSV round-trip (write + bgzip + tabix) takes **{tsv_sec:.2f}s** avg
(**{tsv_pct:.1f}%** of annotation stage time). Total annotation stage: **{annot_sec:.1f}s**.

**Proposed**: Use `pysam.VariantFile` to read the input VCF, add the XGB_PROBA field
directly from the DataFrame, and write the output VCF in a single streaming pass:
```python
import pysam
vcf_in = pysam.VariantFile(input_vcf)
vcf_in.header.add_meta("INFO", items=[...])  # add XGB_PROBA definition
vcf_out = pysam.VariantFile(output_vcf, "wz", header=vcf_in.header)
proba_lookup = dict(zip(keys, predictions))
for record in vcf_in:
    key = (record.chrom, record.pos, record.ref, record.alts[0])
    if key in proba_lookup:
        record.info["XGB_PROBA"] = proba_lookup[key]
    vcf_out.write(record)
```

**Impact**: Eliminates 3 subprocess calls and all intermediate file I/O.
For small annotation data (just one float column), this is likely faster than the
bcftools pipeline.

**Complexity**: Low-Medium -- pysam is already imported in the codebase.
""")

# ─── Recommendation 5: Single-pass streaming with cyvcf2 ────────────────────
read_vcf_sec = _sec("stage_read_vcf_with_aggregation_mean_sec")
read_vcf_pct = _pct("stage_read_vcf_with_aggregation_pct")
recommendations.append(f"""
### 5. Replace ProcessPoolExecutor + subprocess with single-pass streaming

**Current**: VCF-to-DataFrame conversion uses `ProcessPoolExecutor` where each worker
spawns `bcftools query | awk` subprocesses for a genomic region. This creates
O(regions x samples) subprocesses.

**Data**: `read_vcf_with_aggregation` takes **{read_vcf_sec:.1f}s** avg (**{read_vcf_pct:.1f}%** of
pipeline). CPU efficiency suggests significant time is spent waiting for subprocesses.

**Proposed**: If `bcftools query | awk` is replaced with `cyvcf2` (Rec. 2) and awk with
Polars (Rec. 1), the entire conversion becomes pure Python/C. This enables:
- **Single-pass streaming**: Read the VCF once with cyvcf2, accumulate into column arrays
- **Polars lazy evaluation**: Build the DataFrame lazily without intermediate files
- **Thread-based parallelism**: Use Polars' built-in multi-threading instead of ProcessPoolExecutor
  (Polars already uses all available cores for operations like `explode`, `group_by`, etc.)

This eliminates the region-chunking overhead, process pool management, and per-region
Parquet part-files.

**Impact**: Could reduce `read_vcf_with_aggregation` time significantly by removing
all subprocess and inter-process communication overhead.

**Complexity**: High -- requires redesigning the VCF-to-DataFrame pipeline.
This is the most impactful but also the largest change.
""")

# ─── Recommendation 6: Replace bedtools closest with pyranges ────────────────
filter_sec = _sec("stage_filter_and_annotate_tr_mean_sec")
filter_pct = _pct("stage_filter_and_annotate_tr_pct")
recommendations.append(f"""
### 6. Replace `bedtools closest` with `pyranges` or `polars` interval join

**Current**: TR annotation runs:
`bcftools query → bedtools closest → cut → sort → bgzip` as a shell pipe.

**Data**: `filter_and_annotate_tr` takes **{filter_sec:.1f}s** avg (**{filter_pct:.1f}%** of pipeline).
This is a chain of 5 subprocess invocations.

**Proposed**: Use `pyranges` for the nearest-interval lookup:
```python
import pyranges as pr
variants = pr.PyRanges(chromosomes=chroms, starts=starts, ends=ends)
tr_regions = pr.read_bed(ref_tr_file)
nearest = variants.nearest(tr_regions)
```
Or use Polars `join_asof` for sorted positional lookups (if data is position-sorted).

**Impact**: Replaces 5 subprocess invocations with a single Python call.
pyranges uses NCLS (nested containment list) which is O(n log n) and very fast.

**Complexity**: Medium -- requires converting the TR annotation logic to pyranges.
The subsequent `bcftools annotate` step to write TR fields to VCF can also be
replaced with the pysam approach from Recommendation 4.
""")

# ─── Print all recommendations ───────────────────────────────────────────────
report = f"""
# Architectural Recommendations Report

**Pipeline average wall-clock time**: {total_sec:.1f}s per shard

---
{"---".join(recommendations)}
---

## Priority Matrix

| # | Recommendation | Impact | Complexity | Priority |
|---|---------------|--------|------------|----------|
| 1 | Replace awk with Polars list ops | High | Medium | **P0** |
| 2 | Replace bcftools query with cyvcf2 | High | Medium-High | **P0** |
| 3 | Eliminate Polars→Pandas for XGBoost | Low-Medium | Low | **P1** (quick win) |
| 4 | In-memory VCF annotation (pysam) | Medium | Low-Medium | **P1** |
| 5 | Single-pass streaming pipeline | Very High | High | **P0** (combines 1+2) |
| 6 | Replace bedtools with pyranges | Medium | Medium | **P2** |

**Recommended approach**: Implement Rec. 5 (which subsumes Rec. 1 and 2) as the primary
effort -- it addresses the largest bottleneck. In parallel, implement Rec. 3 and 4 as
quick wins. Rec. 6 can follow as a separate cleanup.
"""

ipy_display(Markdown(report))

## Export Data

Save all parsed DataFrames for downstream analysis.

In [ ]:
# ── Save all analysis data ────────────────────────────────────────────────────
export_dir = LOG_CACHE_DIR / "analysis_output"
export_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "perf_events.csv": perf_df,
    "stage_timing.csv": timing_pivot,
    "bottleneck_summary.csv": bottleneck_summary.to_frame("value"),
}
if not sub_read_vcf_df.empty:
    exports["substage_read_vcf.csv"] = sub_read_vcf_df
if not sub_vtp_df.empty:
    exports["substage_vcf_to_parquet.csv"] = sub_vtp_df
if not sub_clf_df.empty:
    exports["substage_classifier.csv"] = sub_clf_df
if not sub_annot_df.empty:
    exports["substage_annotation.csv"] = sub_annot_df
if not cpu_eff_df.empty:
    exports["cpu_efficiency.csv"] = cpu_eff_df
if not run_summary.empty:
    exports["run_summary.csv"] = run_summary

for filename, df in exports.items():
    path = export_dir / filename
    df.to_csv(path, index=True)
    print(f"  Saved {path} ({len(df)} rows)")

print(f"\nAll exports saved to: {export_dir}")